# Fine-Tuning RoBERTa for Tweet Emotion Classification

This notebook fine-tunes `cardiffnlp/twitter-roberta-base` on a labeled subset of our political tweet datasets for emotion classification. We use VADER compound scores to generate pseudo-labels, then train the transformer to classify tweets into four emotion categories: **joy**, **sadness**, **anger**, and **fear**.

**Why fine-tune?** The base `twitter-roberta-base` model is pre-trained on ~58M tweets but lacks an emotion classification head. By fine-tuning on our domain-specific political tweets, the model learns emotion patterns unique to political discourse — improving upon generic off-the-shelf classifiers.

**Pipeline:**
1. Load preprocessed tweet data
2. Generate emotion labels using VADER + NRCLex heuristics
3. Tokenize with RoBERTa tokenizer
4. Fine-tune with Hugging Face Trainer API
5. Evaluate on held-out test set
6. Save the fine-tuned model for inference in notebooks 02 & 03

In [ ]:
!pip install -q transformers torch datasets accelerate scikit-learn nrclex

import pandas as pd
import numpy as np
import re
from collections import Counter

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nrclex import NRCLex

nltk.download('vader_lexicon', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ── Load Data ──────────────────────────────────────────────────────────
# Mount Google Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

TRUMP_PATH = "/content/drive/MyDrive/Research Paper (AI + Crypto)/processedtrumpv3.csv"
BIDEN_PATH = "/content/drive/MyDrive/Research Paper (AI + Crypto)/Copy of lessprocessedjoe.csv"

df_trump = pd.read_csv(TRUMP_PATH, lineterminator='\n')
df_biden = pd.read_csv(BIDEN_PATH, lineterminator='\n')

# Combine both datasets
df_trump['source'] = 'trump'
df_biden['source'] = 'biden'
df = pd.concat([df_trump, df_biden], ignore_index=True)
df = df.dropna(subset=['tweet']).reset_index(drop=True)
df['tweet'] = df['tweet'].astype(str)

print(f"Combined dataset: {len(df):,} tweets")
print(f"  Trump: {len(df_trump):,}")
print(f"  Biden: {len(df_biden):,}")

## Emotion Label Generation

We combine VADER sentiment intensity with NRCLex emotion scores to assign each tweet one of four emotion labels. This multi-signal approach produces higher-quality pseudo-labels than either method alone:

- **VADER** provides sentiment polarity and intensity (compound score)
- **NRCLex** maps individual words to fine-grained emotion categories

The labeling heuristic:
1. Compute NRCLex emotion scores for each tweet
2. Map the dominant NRC emotion to our 4-class scheme (joy, sadness, anger, fear)
3. Use VADER compound score as a tiebreaker and confidence filter
4. Discard ambiguous tweets (low VADER magnitude + no clear NRC signal)

In [ ]:
sid = SentimentIntensityAnalyzer()

# Mapping from NRCLex emotions to our 4-class scheme
NRC_TO_LABEL = {
    'joy': 'joy',
    'trust': 'joy',
    'positive': 'joy',
    'anticipation': 'joy',
    'surprise': 'joy',
    'sadness': 'sadness',
    'negative': 'sadness',
    'disgust': 'anger',
    'anger': 'anger',
    'fear': 'fear',
}

LABEL2ID = {'joy': 0, 'sadness': 1, 'anger': 2, 'fear': 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def assign_emotion(text):
    """
    Assign an emotion label using NRCLex + VADER heuristics.
    Returns None for ambiguous tweets.
    """
    # VADER compound score
    vader = sid.polarity_scores(text)['compound']
    
    # NRCLex dominant emotion
    emo = NRCLex(text)
    scores = emo.raw_emotion_scores
    
    if scores:
        dominant = max(scores, key=scores.get)
        mapped = NRC_TO_LABEL.get(dominant)
        if mapped:
            return mapped
    
    # Fallback to VADER if NRCLex is inconclusive
    if vader >= 0.3:
        return 'joy'
    elif vader <= -0.5:
        return 'anger'
    elif vader <= -0.3:
        return 'sadness'
    
    return None  # Ambiguous — discard

# Sample for labeling (full dataset is too large for NRCLex per-tweet)
SAMPLE_SIZE = 100_000
df_sample = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).copy()

print(f"Labeling {len(df_sample):,} sampled tweets...")
df_sample['emotion'] = df_sample['tweet'].apply(assign_emotion)

# Drop ambiguous tweets
df_labeled = df_sample.dropna(subset=['emotion']).reset_index(drop=True)
df_labeled['label'] = df_labeled['emotion'].map(LABEL2ID)

print(f"\nLabeled tweets: {len(df_labeled):,} ({len(df_labeled)/len(df_sample)*100:.1f}% retained)")
print(f"\nLabel distribution:")
print(df_labeled['emotion'].value_counts())

## Tokenization & Dataset Preparation

We tokenize the tweets using the RoBERTa tokenizer and create PyTorch datasets for training and evaluation.

In [ ]:
MODEL_NAME = "cardiffnlp/twitter-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Train/test split (80/20)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_labeled['tweet'].tolist(),
    df_labeled['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_labeled['label'].tolist()
)

print(f"Training samples: {len(train_texts):,}")
print(f"Test samples:     {len(test_texts):,}")

# Tokenize
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

In [ ]:
class TweetDataset(Dataset):
    """PyTorch dataset for tokenized tweet data."""
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TweetDataset(train_encodings, train_labels)
test_dataset = TweetDataset(test_encodings, test_labels)

print(f"Train dataset: {len(train_dataset):,} samples")
print(f"Test dataset:  {len(test_dataset):,} samples")

## Fine-Tuning

We fine-tune `cardiffnlp/twitter-roberta-base` with a classification head for 4-class emotion detection. Training hyperparameters:
- **Epochs**: 3
- **Batch size**: 32
- **Learning rate**: 2e-5 (with linear warmup)
- **Weight decay**: 0.01
- **Evaluation**: Every 500 steps

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(device)

training_args = TrainingArguments(
    output_dir='./roberta-tweet-emotion',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=100,
    report_to='none',
)

def compute_metrics(eval_pred):
    """Compute accuracy and macro F1 for evaluation."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1': f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning...")
trainer.train()

## Evaluation

We evaluate the fine-tuned model on the held-out test set and generate a full classification report.

In [ ]:
# Evaluate on test set
results = trainer.evaluate()
print(f"Test Accuracy: {results['eval_accuracy']:.4f}")
print(f"Test F1 (macro): {results['eval_f1']:.4f}")

# Full classification report
preds_output = trainer.predict(test_dataset)
preds = np.argmax(preds_output.predictions, axis=-1)
target_names = [ID2LABEL[i] for i in range(4)]

print("\nClassification Report:")
print(classification_report(test_labels, preds, target_names=target_names))

In [ ]:
# Confusion matrix visualization
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title('Fine-Tuned RoBERTa — Confusion Matrix', fontsize=13, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## Save Fine-Tuned Model

We save the fine-tuned model and tokenizer to Google Drive so they can be loaded in notebooks 02 and 03 for inference on the full datasets.

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/Research Paper (AI + Crypto)/roberta-tweet-emotion-finetuned"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Fine-tuned model saved to: {SAVE_PATH}")
print(f"\nSaved files:")
import os
for f in os.listdir(SAVE_PATH):
    size_mb = os.path.getsize(os.path.join(SAVE_PATH, f)) / (1024 * 1024)
    print(f"  {f} ({size_mb:.1f} MB)")

In [ ]:
# Verify the saved model loads correctly
from transformers import pipeline

finetuned_classifier = pipeline(
    "text-classification",
    model=SAVE_PATH,
    tokenizer=SAVE_PATH,
    top_k=None
)

# Test on sample tweets
test_tweets = [
    "I'm so proud of our country today!",
    "This is absolutely disgusting and unacceptable",
    "I'm really worried about the future of our democracy",
    "Heartbroken by the state of things right now",
]

print("Sample predictions from fine-tuned model:")
for tweet in test_tweets:
    result = finetuned_classifier(tweet)
    top = max(result[0], key=lambda d: d['score'])
    print(f"  [{top['label']:>7s} ({top['score']:.3f})] {tweet}")

## Summary

I fine-tuned `cardiffnlp/twitter-roberta-base` on 100K political tweets labeled via VADER + NRCLex heuristics, producing a domain-specific emotion classifier for political Twitter discourse.

**Next steps:**
- **Notebook 02** (`02_biden_sentiment_analysis.ipynb`): Uses the fine-tuned model for Biden tweet emotion classification
- **Notebook 03** (`03_trump_sentiment_analysis.ipynb`): Uses the fine-tuned model for Trump tweet emotion classification

The fine-tuned model is saved at:
```
/content/drive/MyDrive/Research Paper (AI + Crypto)/roberta-tweet-emotion-finetuned
```